# Network Security Analyzer

> **Authorized lab use only.** This notebook is an educational demonstration. Run it only with synthetic data or systems you own and are permitted to test.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import hashlib
import secrets
import base64
import socket
import struct
import re
from datetime import datetime, timedelta
import ipaddress
from collections import Counter
import warnings
warnings.filterwarnings('ignore')

class SecurityAnalyzer:
    def __init__(self):
        self.threat_patterns = [
            r'\b(?:\d{1,3}\.){3}\d{1,3}\b',
            r'[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}',
            r'(?:http|https|ftp)://[^\s/$.?#].[^\s]*',
            r'\b(?:SELECT|INSERT|UPDATE|DELETE|DROP|UNION)\b',
            r'<script[^>]*>.*?</script>',
            r'eval\s*\(',
            r'document\.cookie',
            r'javascript:',
            r'\.\./|\.\.\\'
        ]

    def generate_sample_logs(self, num_entries=1000):
        np.random.seed(42)

        ips = [f"{np.random.randint(1,255)}.{np.random.randint(1,255)}.{np.random.randint(1,255)}.{np.random.randint(1,255)}"
               for _ in range(num_entries)]

        attack_types = ['bruteforce', 'sqli', 'xss', 'ddos', 'malware', 'phishing', 'normal']
        severities = ['low', 'medium', 'high', 'critical']

        logs = []
        for i in range(num_entries):
            timestamp = datetime.now() - timedelta(days=np.random.randint(0, 30))
            ip = ips[i]
            attack = np.random.choice(attack_types, p=[0.05, 0.08, 0.06, 0.03, 0.04, 0.04, 0.7])
            severity = np.random.choice(severities, p=[0.4, 0.3, 0.2, 0.1])
            port = np.random.choice([22, 80, 443, 21, 23, 25, 53, 110, 143])

            logs.append({
                'timestamp': timestamp,
                'source_ip': ip,
                'attack_type': attack,
                'severity': severity,
                'port': port,
                'bytes_transferred': np.random.randint(100, 10000)
            })

        return pd.DataFrame(logs)

    def detect_anomalies(self, df):
        anomalies = []

        ip_counts = df['source_ip'].value_counts()
        suspicious_ips = ip_counts[ip_counts > ip_counts.quantile(0.95)].index.tolist()

        attack_ips = df[df['attack_type'] != 'normal']['source_ip'].unique()

        high_traffic_ports = df.groupby('port')['bytes_transferred'].sum()
        suspicious_ports = high_traffic_ports[high_traffic_ports > high_traffic_ports.quantile(0.9)].index.tolist()

        return {
            'suspicious_ips': suspicious_ips,
            'attack_source_ips': attack_ips.tolist(),
            'high_traffic_ports': suspicious_ports
        }

    def password_strength_analysis(self, passwords):
        results = []

        for pwd in passwords:
            score = 0
            feedback = []

            if len(pwd) >= 8:
                score += 1
            else:
                feedback.append("Too short")

            if re.search(r'[A-Z]', pwd):
                score += 1
            else:
                feedback.append("No uppercase")

            if re.search(r'[a-z]', pwd):
                score += 1
            else:
                feedback.append("No lowercase")

            if re.search(r'\d', pwd):
                score += 1
            else:
                feedback.append("No numbers")

            if re.search(r'[!@#$%^&*(),.?":{}|<>]', pwd):
                score += 1
            else:
                feedback.append("No special chars")

            strength = ['Very Weak', 'Weak', 'Fair', 'Good', 'Strong'][min(score, 4)]

            results.append({
                'password': pwd,
                'strength': strength,
                'score': score,
                'feedback': feedback
            })

        return results

    def hash_analysis(self, text_list):
        results = []

        for text in text_list:
            md5_hash = hashlib.md5(text.encode()).hexdigest()
            sha1_hash = hashlib.sha1(text.encode()).hexdigest()
            sha256_hash = hashlib.sha256(text.encode()).hexdigest()

            results.append({
                'original': text,
                'md5': md5_hash,
                'sha1': sha1_hash,
                'sha256': sha256_hash
            })

        return results

    def network_scan_simulation(self, target_range="192.168.1.0/24"):
        try:
            network = ipaddress.ip_network(target_range, strict=False)
            scan_results = []

            for i, ip in enumerate(list(network.hosts())[:20]):
                if np.random.random() > 0.7:
                    open_ports = np.random.choice([22, 23, 80, 443, 21, 25],
                                                size=np.random.randint(1, 4), replace=False)
                    scan_results.append({
                        'ip': str(ip),
                        'status': 'up',
                        'open_ports': open_ports.tolist(),
                        'os_guess': np.random.choice(['Linux', 'Windows', 'macOS'])
                    })
                else:
                    scan_results.append({
                        'ip': str(ip),
                        'status': 'down',
                        'open_ports': [],
                        'os_guess': 'Unknown'
                    })

            return scan_results
        except:
            return []

    def vulnerability_assessment(self, services):
        vulnerabilities = {
            22: ['SSH Weak Ciphers', 'Default Credentials'],
            23: ['Telnet Cleartext', 'Buffer Overflow'],
            80: ['SQL Injection', 'XSS', 'Directory Traversal'],
            443: ['SSL/TLS Misconfiguration', 'Weak Ciphers'],
            21: ['FTP Anonymous Login', 'Buffer Overflow'],
            25: ['SMTP Open Relay', 'Header Injection']
        }

        assessment = []
        for service in services:
            port = service.get('port', 80)
            if port in vulnerabilities:
                vulns = np.random.choice(vulnerabilities[port],
                                       size=np.random.randint(1, len(vulnerabilities[port])+1),
                                       replace=False)
                assessment.append({
                    'service': f"Port {port}",
                    'vulnerabilities': vulns.tolist(),
                    'risk_level': np.random.choice(['Low', 'Medium', 'High', 'Critical'])
                })

        return assessment

    def generate_security_report(self, df):
        report = {
            'total_events': len(df),
            'attack_events': len(df[df['attack_type'] != 'normal']),
            'unique_ips': df['source_ip'].nunique(),
            'attack_distribution': df['attack_type'].value_counts().to_dict(),
            'severity_distribution': df['severity'].value_counts().to_dict(),
            'top_target_ports': df['port'].value_counts().head().to_dict(),
            'time_range': f"{df['timestamp'].min()} to {df['timestamp'].max()}"
        }

        return report

def create_visualizations(analyzer, df):
    fig, axes = plt.subplots(2, 3, figsize=(18, 12))

    attack_counts = df['attack_type'].value_counts()
    axes[0,0].pie(attack_counts.values, labels=attack_counts.index, autopct='%1.1f%%')
    axes[0,0].set_title('Attack Type Distribution')

    severity_counts = df['severity'].value_counts()
    axes[0,1].bar(severity_counts.index, severity_counts.values,
                  color=['green', 'yellow', 'orange', 'red'])
    axes[0,1].set_title('Severity Levels')
    axes[0,1].tick_params(axis='x', rotation=45)

    hourly_attacks = df.set_index('timestamp').resample('H').size()
    axes[0,2].plot(hourly_attacks.index, hourly_attacks.values)
    axes[0,2].set_title('Attacks Over Time')
    axes[0,2].tick_params(axis='x', rotation=45)

    port_attacks = df['port'].value_counts().head(10)
    axes[1,0].barh(range(len(port_attacks)), port_attacks.values)
    axes[1,0].set_yticks(range(len(port_attacks)))
    axes[1,0].set_yticklabels(port_attacks.index)
    axes[1,0].set_title('Top Targeted Ports')

    ip_attacks = df['source_ip'].value_counts().head(10)
    axes[1,1].barh(range(len(ip_attacks)), ip_attacks.values)
    axes[1,1].set_yticks(range(len(ip_attacks)))
    axes[1,1].set_yticklabels([ip[:10] + '...' for ip in ip_attacks.index])
    axes[1,1].set_title('Top Source IPs')

    heatmap_data = df.pivot_table(values='bytes_transferred',
                                  index='attack_type',
                                  columns='severity',
                                  aggfunc='mean',
                                  fill_value=0)
    sns.heatmap(heatmap_data, annot=True, fmt='.0f', ax=axes[1,2])
    axes[1,2].set_title('Average Bytes by Attack & Severity')

    plt.tight_layout()
    plt.show()

def run_comprehensive_analysis():
    print("🔒 CYBERSECURITY ANALYSIS SUITE")
    print("=" * 50)

    analyzer = SecurityAnalyzer()

    print("\n📊 Generating sample security logs...")
    df = analyzer.generate_sample_logs(1500)
    print(f"Generated {len(df)} log entries")

    print("\n🚨 Detecting anomalies...")
    anomalies = analyzer.detect_anomalies(df)
    print(f"Found {len(anomalies['suspicious_ips'])} suspicious IPs")
    print(f"Found {len(anomalies['attack_source_ips'])} attack source IPs")

    print("\n🔐 Password strength analysis...")
    test_passwords = ['123456', 'password', 'P@ssw0rd123', 'admin', 'Tr0ub4dor&3']
    pwd_results = analyzer.password_strength_analysis(test_passwords)

    for result in pwd_results:
        print(f"Password: {result['password'][:8]}... | Strength: {result['strength']}")

    print("\n🔍 Hash analysis...")
    test_strings = ['admin', 'password123', 'secret_key']
    hash_results = analyzer.hash_analysis(test_strings)

    for result in hash_results:
        print(f"Text: {result['original']} | SHA256: {result['sha256'][:16]}...")

    print("\n🌐 Network scan simulation...")
    scan_results = analyzer.network_scan_simulation()
    active_hosts = [r for r in scan_results if r['status'] == 'up']
    print(f"Found {len(active_hosts)} active hosts")

    print("\n⚠️  Vulnerability assessment...")
    services = [{'port': 22}, {'port': 80}, {'port': 443}]
    vuln_assessment = analyzer.vulnerability_assessment(services)

    for assessment in vuln_assessment:
        print(f"Service: {assessment['service']} | Risk: {assessment['risk_level']}")

    print("\n📋 Security report...")
    report = analyzer.generate_security_report(df)

    print(f"Total Events: {report['total_events']}")
    print(f"Attack Events: {report['attack_events']}")
    print(f"Attack Rate: {(report['attack_events']/report['total_events']*100):.1f}%")
    print(f"Unique Source IPs: {report['unique_ips']}")

    print("\n📈 Creating visualizations...")
    create_visualizations(analyzer, df)

    print("\n🎯 Top Threats Summary:")
    attack_dist = report['attack_distribution']
    for attack_type, count in sorted(attack_dist.items(), key=lambda x: x[1], reverse=True):
        if attack_type != 'normal':
            print(f"  {attack_type.upper()}: {count} incidents")

    print("\n✅ Analysis complete!")

    return analyzer, df, report

def advanced_threat_hunting(df):
    print("\n🕵️ ADVANCED THREAT HUNTING")
    print("=" * 30)

    threat_indicators = []

    high_volume_ips = df.groupby('source_ip').size()
    high_volume_ips = high_volume_ips[high_volume_ips > high_volume_ips.quantile(0.9)]
    threat_indicators.extend([f"High volume IP: {ip}" for ip in high_volume_ips.index])

    unusual_ports = df[~df['port'].isin([80, 443, 22])]['port'].value_counts()
    threat_indicators.extend([f"Unusual port activity: {port}" for port in unusual_ports.index[:5]])

    attack_sequences = df[df['attack_type'] != 'normal'].groupby('source_ip')['attack_type'].apply(list)
    multi_attack_ips = attack_sequences[attack_sequences.apply(len) > 2]
    threat_indicators.extend([f"Multi-vector attack from: {ip}" for ip in multi_attack_ips.index[:3]])

    print(f"🔍 Found {len(threat_indicators)} threat indicators:")
    for indicator in threat_indicators[:10]:
        print(f"  ⚠️  {indicator}")

    return threat_indicators

if __name__ == "__main__":
    analyzer, df, report = run_comprehensive_analysis()
    threat_indicators = advanced_threat_hunting(df)

    print(f"\n🎉 Cybersecurity analysis completed successfully!")
    print(f"📊 Analyzed {len(df)} security events")
    print(f"🚨 Identified {len(threat_indicators)} potential threats")